In [2]:
// adapter_mapping_example.cs
using System;
using System.Text.Json;

// =========================================================
// GENERIC ADAPTER CONTRACT
// =========================================================

public interface IAdapter<TSource, TTarget>
{
	TTarget Map(TSource source);
}

// =========================================================
// SKYTRANSFER OUTPUT MODEL
// =========================================================

public class SkyTransferPromotion
{
	public string Name { get; set; }

	public decimal DiscountValue { get; set; }

	public string DiscountType { get; set; }

	public DateTime StartTime { get; set; }

	public DateTime? EndTime { get; set; }
}

// =========================================================
// SKYCRM DOMAIN MODEL
// =========================================================

public class SkyCrmPromotion
{
	public string Name { get; set; }

	public decimal Value { get; set; }

	public string Type { get; set; }

	public DateTime Start { get; set; }

	public DateTime? End { get; set; }
}

// =========================================================
// RK7 REQUEST MODEL
// =========================================================

public class RK7PromotionRequest
{
	public string PromotionName { get; set; }

	public decimal Amount { get; set; }

	public string Mode { get; set; }

	public string BeginDate { get; set; }

	public string EndDate { get; set; }
}

// =========================================================
// REFAGENT REQUEST MODEL
// =========================================================

public class RefAgentPromotionRequest
{
	public string Code { get; set; }

	public decimal Discount { get; set; }

	public string PromotionType { get; set; }

	public long StartTimestamp { get; set; }

	public long? EndTimestamp { get; set; }
}

// =========================================================
// SKYTRANSFER → SKYCRM ADAPTER
// =========================================================

public class SkyTransferToSkyCrmAdapter 
	: IAdapter<SkyTransferPromotion, SkyCrmPromotion>
{
	public SkyCrmPromotion Map(SkyTransferPromotion src)
	{
		return new SkyCrmPromotion
		{
			Name = src.Name,
			Value = src.DiscountValue,
			Type = src.DiscountType,
			Start = src.StartTime,
			End = src.EndTime
		};
	}
}

// =========================================================
// SKYCRM → RK7 ADAPTER
// =========================================================

public class SkyCrmToRK7Adapter 
	: IAdapter<SkyCrmPromotion, RK7PromotionRequest>
{
	public RK7PromotionRequest Map(SkyCrmPromotion src)
	{
		return new RK7PromotionRequest
		{
			PromotionName = src.Name,
			Amount = src.Value,
			Mode = src.Type,
			BeginDate = src.Start.ToString("yyyy-MM-dd"),
			EndDate = src.End?.ToString("yyyy-MM-dd")
		};
	}
}

// =========================================================
// SKYCRM → REFAGENT ADAPTER
// =========================================================

public class SkyCrmToRefAgentAdapter 
	: IAdapter<SkyCrmPromotion, RefAgentPromotionRequest>
{
	public RefAgentPromotionRequest Map(SkyCrmPromotion src)
	{
		return new RefAgentPromotionRequest
		{
			Code = src.Name,
			Discount = src.Value,
			PromotionType = src.Type,
			StartTimestamp = 
				new DateTimeOffset(src.Start).ToUnixTimeSeconds(),

			EndTimestamp =
				src.End == null
					? null
					: new DateTimeOffset(src.End.Value)
						.ToUnixTimeSeconds()
		};
	}
}

// =========================================================
// DEMO
// =========================================================

var skyTransfer = new SkyTransferPromotion
{
	Name = "promo 10%",
	DiscountValue = 10,
	DiscountType = "PERCENT",
	StartTime = DateTime.UtcNow
};

// 1. skytransfer → skycrm
var skyTransferAdapter = new SkyTransferToSkyCrmAdapter();
var crmPromotion = skyTransferAdapter.Map(skyTransfer);
Console.WriteLine($"1: {JsonSerializer.Serialize(crmPromotion)}");

// 2. skycrm → rk7
var rk7Adapter = new SkyCrmToRK7Adapter();
var rk7Request = rk7Adapter.Map(crmPromotion);
Console.WriteLine($"2: {JsonSerializer.Serialize(rk7Request)}");

// 3. skycrm → refagent
var refAgentAdapter = new SkyCrmToRefAgentAdapter();
var refAgentRequest = refAgentAdapter.Map(crmPromotion);
Console.WriteLine($"3: {JsonSerializer.Serialize(refAgentRequest)}");
Console.WriteLine("mapping done");


1: {"Name":"promo 10%","Value":10,"Type":"PERCENT","Start":"2026-03-05T10:17:02.2515401Z","End":null}
2: {"PromotionName":"promo 10%","Amount":10,"Mode":"PERCENT","BeginDate":"2026-03-05","EndDate":null}
3: {"Code":"promo 10%","Discount":10,"PromotionType":"PERCENT","StartTimestamp":1772705822,"EndTimestamp":null}
mapping done


## flow mapping thực tế
```
skytransfer promotion
        ↓
SkyTransferToSkyCrmAdapter
        ↓
skycrm promotion (domain)
       / \
      /   \
rk7 adapter   refagent adapter
      ↓            ↓
rk7 request   refagent request
```

## cấu trúc
```
integrations
├─ skytransfer
│   └─ promotions
│       ├─ models
│       └─ adapters
```